# XRZ — Agent-turn Merkle breakpoints, v2 successor

**Synthetic fixture only.** The v2 fixture includes typed instruction/code/test events and an explicit predecessor reference to the retained v1 fixture root. Independent local reconstruction and negative controls are tested. No real agent execution is asserted by the fixture.

## Method

Source `custody/agent_turn_mmr.py`; v2 schema `contracts/AGENT_TURN.v2.schema.json`; predecessor schema v1 preserved. The predecessor root is a **synthetic reference fixture**, not an authenticated real-world chain. A hash cannot establish authorship, scientific correctness, or BSL-2 clearance.

In [ ]:
from pathlib import Path
import sys, copy, json
repo = Path.cwd().resolve()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))
from custody.agent_turn_mmr import construct_turn_checkpoint, reference_verify
print("Local reference file:", repo / "custody" / "agent_turn_mmr.py")

## Input observations — synthetic, not actual GitHub Copilot telemetry

In [ ]:
events = []
kinds = ["INSTRUCTION_READ", "CODE_EDIT", "TEST_EXECUTION"]
labels = ["synthetic_instructions_read","synthetic_code_proposal","synthetic_test_failure"]
for i, (kind,label) in enumerate(zip(kinds,labels)):
    events.append({"schema":"xrz.agent_turn.v2",
       "event_kind":kind,
       "event_id":"SYNTHETIC-V2-"+str(i),
       "actor":"SYNTHETIC_COPILOT",
       "turn_id":"synthetic-turn-002","repo":"biobitworks/xrz",
       "task_ref":"synthetic-local-successor-fixture","sequence":i,
       "status":"SYNTHETIC",
       "input_occurrences":["source-occurrence-0"],
       "output_occurrences":["fixture-v2-occurrence-"+str(i)],
       "evidence_refs":["synthetic:"+label]})
assert len(events)==3
print("Typed v2 fixture count:",len(events))

## Deterministic construction and dual-codepath verification

In [ ]:
checkpoint=construct_turn_checkpoint(events, parent_checkpoint="0fc4c6332e1c1831036193a6d66b235e7b2fc5893908dd49c880a32fb234166c")
assert checkpoint["root"] == "d459eb5b1c36a67b8637a4897e6201ae5e8dff7ccd47e03434a792b1968e0049", checkpoint["root"]
assert reference_verify(events,checkpoint)
print(json.dumps({"root":checkpoint["root"],"peaks":checkpoint["peaks"],"leaf_count":checkpoint["event_count"],"signature_state":checkpoint["signature_state"],"state":checkpoint["state"],"local_verifier":"PASS_SINGLE_HOST"},indent=2))

## Negative controls — deliberately corrupted input

In [ ]:
changed=copy.deepcopy(events)
changed[1]["status"]="OBSERVED"
reordered=copy.deepcopy(events)
reordered[0],reordered[1]=reordered[1],reordered[0]
wrong_parent=dict(checkpoint,parent_checkpoint="cd"*32)
checks={"mutation_rejected":not reference_verify(changed,checkpoint),
"reorder_rejected":not reference_verify(reordered,checkpoint),
"truncation_rejected":not reference_verify(events[:-1],checkpoint),
"parent_substitution_rejected":not reference_verify(events,wrong_parent)}
assert all(checks.values()),checks
print(json.dumps(checks,indent=2))

## Takeaways and NEXT_ACTION

- The only committed fixture here is **synthetic**. The SHA and peaks commit ordered leaf contents and a parent-root reference, not authorship, science or genuine Copilot activity.
- **Reproduction:** from repository root run `PYTHONPATH=. python -m unittest discover -s tests -v` then `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/00_agent_turn_merkle_breakpoints.ipynb` after installing `nbformat nbclient ipykernel`.
- **Known gaps:** two-host verification, authenticated parent history and append-only consistency, actual Copilot task/review receipt, signatures, mobile and XR tests, biosafety approvals.
- **NEXT_ACTION:** request real Copilot PR review and capture externally source-bound event occurrences before promoting L1 to L2 project custody.